# asyncio

对应 `stdlib.md`：异步 I/O。

笔记本在 `python_base/asyncio/qa.ipynb`。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。


In [2]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录，即 python_base/asyncio。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "asyncio" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "python_base" / "asyncio"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/python_base/asyncio')

## 1. 跑一个协程

写协程 `ping`，返回 `"pong"`。用 `asyncio.run` 跑 `ping()`，打印返回值。


In [11]:
# 作答

import asyncio

async def ping():
    return "pong"

# can't use in Jupyter
# result = asyncio.run(ping())

result = await ping()
print(result)

#评阅
# 对。协程返回 pong。Jupyter 里事件循环已经在跑，asyncio.run 会报错，改用 await 是对的。脚本里才写 asyncio.run(ping())。

#参考答案
# import asyncio
# async def ping():
#     return "pong"
# print(await ping())


pong


## 2. 中间停一下

写一个协程：先打印 `before`，用 `asyncio.sleep` 等 `0.1` 秒，再打印 `after`。用 `asyncio.run` 跑它。


In [6]:
# 作答

import asyncio

async def ping():
    print("before")
    await asyncio.sleep(0.1)
    print("after")

# can't use in Jupyter
# result = asyncio.run(ping())

await ping()

#评阅
# 对。先 before，sleep 之后才 after。和上一题一样，笔记本里用 await，不用 asyncio.run。

#参考答案
# import asyncio
# async def tick():
#     print("before")
#     await asyncio.sleep(0.1)
#     print("after")
# await tick()


before
after


## 3. 一起等，顺序跟传入的一样

`fetch` 已经写好。用 `asyncio.gather` 同时跑 `fetch("slow")` 和 `fetch("fast")`，打印得到的列表。


In [9]:
import asyncio

async def fetch(name: str) -> str:
    delay = {"slow": 0.2, "fast": 0.05}[name]
    await asyncio.sleep(delay)
    return name

# 作答

result = await asyncio.gather(
    fetch("slow"),
    fetch("fast"),
)
print(result)

#评阅
# 对。gather 按传入顺序给出 slow、fast，不是谁先做完谁在前。

#参考答案
# print(await asyncio.gather(fetch("slow"), fetch("fast")))


['slow', 'fast']


## 4. 先启动，再取结果

`load` 已经写好。写一个协程：用 `asyncio.create_task` 启动 `load()`，接着打印 `started`，再打印任务的返回值。用 `asyncio.run` 跑这个协程。


In [12]:
import asyncio

async def load() -> str:
    await asyncio.sleep(0.1)
    return "done"

# 作答

task = asyncio.create_task(load())
print("started")
result = await task
print(result)

#评阅
# 对。create_task 启动后先打印 started，再取到 done。脚本里要放进协程再用 asyncio.run；笔记本里顶层这样写可以。

#参考答案
# async def main():
#     task = asyncio.create_task(load())
#     print("started")
#     print(await task)
# await main()


started
done


## 5. 等太久就放弃

`slow` 会等 1 秒。用 `asyncio.wait_for` 最多等 `0.05` 秒。任务会超时，打印异常类型的名字。


In [13]:
import asyncio

async def slow() -> str:
    await asyncio.sleep(1)
    return "late"

# 作答

try:
    result = await asyncio.wait_for(slow(), 0.05)
except Exception as e:
    print(type(e).__name__)


#评阅
# 对。wait_for 超时抛出的是 TimeoutError。可以只抓 TimeoutError，不必用 Exception。

#参考答案
# try:
#     await asyncio.wait_for(slow(), 0.05)
# except TimeoutError:
#     print("TimeoutError")


TimeoutError


## 6. 谁先做完谁先拿

`work` 和 `jobs` 已经写好。用 `asyncio.as_completed` 按完成的先后打印每个任务返回的名字。


In [ ]:
import asyncio

async def work(name: str, seconds: float) -> str:
    await asyncio.sleep(seconds)
    return name

jobs = [("slow", 0.3), ("fast", 0.05), ("mid", 0.15)]

# 作答

tasks = [asyncio.create_task(work(name, seconds)) for name, seconds in jobs]
async for task in asyncio.as_completed(tasks):
    print(task.result())

#评阅
# 对，顺序是 fast、mid、slow。普通 for 会多出一个没人 await 的协程，所以有 RuntimeWarning。改成 async for，用 task.result() 取结果，警告就没了。

#参考答案
# tasks = [asyncio.create_task(work(name, seconds)) for name, seconds in jobs]
# async for task in asyncio.as_completed(tasks):
#     print(task.result())


<coroutine object _AsCompletedIterator._wait_for_one at 0x1039966c0>
<coroutine object _AsCompletedIterator._wait_for_one at 0x103995d20>
<coroutine object _AsCompletedIterator._wait_for_one at 0x103996260>


/var/folders/6g/9chj8ddx1033kwmzkbfd95l40000gn/T/ipykernel_68626/1168430953.py:15: RuntimeWarning: coroutine '_AsCompletedIterator._wait_for_one' was never awaited
  for result in asyncio.as_completed(works):
